In [1]:
import pandas as pd

df = pd.read_csv("outputs/crashes_clean.csv")
df

,collision_index,latitude,longitude,vehicle_type,severity_weight,accident_year
0,2021170H10421,54.689833,-1.270905,car,2,2021
1,2021170H10421,54.689833,-1.270905,car,2,2021
2,2021170H11231,54.690592,-1.218333,car,1,2021
3,2021170H11231,54.690592,-1.218333,car,1,2021
4,2020170M11750,54.570397,-1.232884,cycle,1,2020
...,...,...,...,...,...,...
888960,202263D058922,51.998470,-3.284127,car,1,2022
888961,202363D007623,52.630192,-3.163807,car,2,2023
888962,2024631460061,52.703260,-3.179130,car,4,2024
888963,2024631460061,52.703260,-3.179130,car,4,2024


In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from pathlib import Path

OUTPUT_DIR = Path("outputs")

# 1km box — Manchester city centre (British National Grid, metres)
BBOX = (383000, 397500, 384000, 398500)  # (minx, miny, maxx, maxy)

# Pass bbox directly to read_file — pyogrio filters at read time, never loads all 3.9M rows
seg_box   = gpd.read_file(OUTPUT_DIR / "segments.gpkg", bbox=BBOX)
crashes   = pd.read_csv(OUTPUT_DIR / "crashes_segmented.csv")
crash_box = crashes[crashes["segment_id"].isin(seg_box["segment_id"])]

print(f"Segments: {len(seg_box):,}  |  Crashes: {len(crash_box):,}")

crash_gdf = gpd.GeoDataFrame(
    crash_box,
    geometry=gpd.points_from_xy(crash_box["longitude"], crash_box["latitude"]),
    crs="EPSG:4326",
)

m = seg_box.to_crs("EPSG:4326").explore(
    column="road_class",
    tooltip=["segment_id", "road_class", "length_m"],
    name="Segments",
    style_kwds={"weight": 2, "opacity": 0.7},
)
crash_gdf.explore(
    m=m,
    column="vehicle_type",
    tooltip=["vehicle_type", "severity_weight", "accident_year"],
    name="Crashes",
    marker_kwds={"radius": 4, "fill": True},
)
folium.LayerControl().add_to(m)

m.save("map.html")
print("Saved map.html")
m